# 03. SFT/RL curriculum 진단

목표: 논문의 표 수치를 재계산하고, SFT 원자 coverage와 SFT–RL composition overlap을 함께 보는 간단한 curriculum 점검기를 만듭니다.

In [ ]:
table1 = {
    "SFT baseline": 4.8,
    "SFT+RL atomic": 14.8,
    "SFT+RL compound": 42.6,
}
baseline = table1["SFT baseline"]
for name, accuracy in table1.items():
    print(f"{name:16s}: accuracy={accuracy:4.1f}%, gain={accuracy-baseline:+4.1f}%p")
print("compound advantage over atomic RL:", table1["SFT+RL compound"] - table1["SFT+RL atomic"], "%p")

## SFT 원자 누락의 영향

Table 2의 unseen 정확도만 사용해 RL gain과 원자 누락 penalty를 계산합니다.

In [ ]:
table2 = {
    "full_sft": {"sft": (0.30, 0.22), "rl": (0.67, 0.55)},
    "missing_atoms": {"sft": (0.24, 0.15), "rl": (0.52, 0.36)},
}
for setting, scores in table2.items():
    gains = tuple(r - s for r, s in zip(scores["rl"], scores["sft"]))
    print(setting, "RL gains at depths 2/3:", gains)
penalty = tuple(a - b for a, b in zip(table2["full_sft"]["rl"], table2["missing_atoms"]["rl"]))
print("post-RL penalty from missing SFT atoms:", penalty)

## Curriculum 점검기

좋은 curriculum의 필요조건을 단순화합니다: SFT가 전체 atom inventory를 덮고, RL composition이 SFT와 완전히 겹치지 않으며, RL이 새로운 local interface를 제공해야 합니다.

In [ ]:
def curriculum_report(all_atoms, sft_atoms, sft_compositions, rl_compositions):
    missing_atoms = set(all_atoms) - set(sft_atoms)
    overlap = set(sft_compositions) & set(rl_compositions)
    union = set(sft_compositions) | set(rl_compositions)
    jaccard = len(overlap) / len(union) if union else 0.0
    novel_rl = set(rl_compositions) - set(sft_compositions)
    return {
        "sft_atom_coverage": 1 - len(missing_atoms) / len(set(all_atoms)),
        "missing_atoms": sorted(missing_atoms),
        "sft_rl_jaccard": round(jaccard, 3),
        "novel_rl_compositions": sorted(novel_rl),
        "passes_basic_gate": not missing_atoms and bool(novel_rl),
    }

atoms = {"reverse", "rotate", "duplicate", "trim"}
report = curriculum_report(
    atoms,
    sft_atoms=atoms,
    sft_compositions={"reverse>rotate", "duplicate>trim"},
    rl_compositions={"rotate>duplicate", "trim>reverse"},
)
for key, value in report.items():
    print(f"{key}: {value}")

## 해석의 한계

낮은 overlap 자체가 성공을 보장하지 않습니다. RL 조합은 현재 policy가 도달할 수 있어야 하고, reward가 유용한 interface를 구별해야 하며, 안전성과 reward hacking도 별도로 평가해야 합니다. 이 점검기는 데이터 목록을 검토하는 첫 단계일 뿐입니다.